In [16]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns

# Problem Set 4
## Monte Carlo Eksperimenter

I ugeseddel 4 skal I arbejde med et simulationsstudie, hvor I skal undersøge og sammenligne en simpel og multipel OLS estimator i et tilfælde med udeladt variabelbias. Generelt kan I bruge Monte Carlo eksperimenter til at opstille en model, hvor I kan kontrollere alle aspekter af den, undersøge egenskaber ved en estimator, f.eks. hvis MLR.1 til MLR.4 ikke er opfyldt, eller hvis I ønsker at undersøge en ny estimator eller undersøge asymptotiske resultater.

I denne ugeseddel skal I se nærmere på følgende datagenererende proces (DGP): 

\begin{align}
y_i &= \beta_0 + \beta_1 x_{1i} +\beta_2 x_{2i}+ u_i, \\
\beta_0 &= 1,\, \beta_1 = 2, \beta_2=-3, \\
x_1 &\sim N(25,25),\, u\sim U(-50,50), \, \, x_2^*\sim U(10,30) \\
x_2 &= \rho x_1 + x_2^*, \, \rho=0.5, \, n=50
\end{align}

Den datagenerende proces specificerer de sande værdier af $\beta_0$, $\beta_1$ og $\beta_2$ samt fordelingerne af $x_1$, $x_2$ og $u$. I dette tilfælde viser ligning (2) de sande parameterværdier, ligning (3) viser hvilken fordeling $x_1$, $x_2^*$ og $u$ er trukket fra, og ligning (1) viser sammenhængen imellem dem i en lineær regression. Ligning (4) viser, at $x_2$ er en lineær funktion af de stokastiske variable $x_1$ og $x_2^*$, hvor $\rho$ angiver hvor meget $x_2$ afhænger af $x_1$.

I skal undersøge og sammenligne egenskaberne for OLS estimatoren i en simpel og multipel regressionsmodel. Nedenfor er udtrykket for OLS estimatoren for $\beta_1$ i det simple tilfælde, $\widetilde{\beta}_1$, og i et tilfælde med flere forklarende variable, $\widehat{\beta}_1$.

\begin{align} \tag{5}
\widetilde{\beta }_{1}& =\frac{\sum_{i}(x_{i1}-\bar{x}_{1})(y_{i}-\bar{y})}{%
\sum_{i}(x_{i1}-\bar{x}_{1})^{2}} \\[8pt]
\widehat{\beta }& =\left( 
\begin{array}{c} \tag{6}
\widehat{\beta }_{0} \\ 
\widehat{\beta }_{1} \\ 
\widehat{\beta }_{2}%
\end{array}%
\right) =(X^{\prime }X)^{-1}X^{\prime }y
\end{align}%
hvor $X$ i ligning (6) indeholder information om $x_{1}$, $x_{2}$ og en konstant. 


## Gruppespørgsmål

**Opgave 1:** Forventer I en positiv eller negativ bias, når I estimerer en simpel model kun med $x_1$? Hvad er størrelsen på den asymptotiske bias? Beregn dette (brug pen og papir).

[Anvend følgende udtryk for asymptotisk bias, hvor $\widetilde{u}=\beta_2x_2 + u$]
\begin{align*}
p\text{lim}(\widetilde{\beta}_1)-\beta_1 = \frac{\text{Cov}(x_1,\widetilde{u})}{%
\text{Var}(x_1)}
\end{align*}


**Dit svar:**

Negativ bias da beta1 < 0 og corr(x_1,x_2) > 0. asymptotisk bias er -1.5 da beta2 * rho = -3 * 0.5 = -1.5

**Opgave 2:** Opfylder den datagenererende proces MLR.1 til MLR.4? 

**Dit svar:**

MLR. 1 linær sammenhæng ligning 1. 
MLR. 2 Tilfældige og uafhængige obsavetioner ja bliver genereret tilfældigt
MLR. 3 ingen multikollinearitet.
MLR. 4, E[u|X]= 0. ja i lingng 3 bliver de trukket i forskelle fordelinger uafhængigt af hinnadnen. 
 E[u] = -50 +50 / 2 = 0, da det bliver trukket fra en uniform fordeling med -50 50.
 Dermed bliver estimatet middelret







**Opgave 3:** Hvordan kan et Monte Carlo-eksperiment blive brugt til at sammenligne variansen fra to forskellige middelrette estimatorer?

**Dit svar:**

Man bruger de samme udtrækninger og kan beregne variansen af de to estimatorer dermed vil man vælge den med laveste varians.

## Python øvelser

### Opgave 1

I skal starte med at opstille Monte Carlo eksperimentet i Python. I skal udfylde den del af Python-koden, som mangler. I skal først køre Monte Carlo eksperimentet, når I har skrevet al koden ind. Der skal udfyldes manglende kode i hvert af de fire steps angivet i kommentarerne til funktionen herunder.

In [17]:
def simulate():
	## Step 1. Definer parameterværdier
	n = 50
	rho = 0.5
	beta0 = 1
	beta1 = 2
	beta2 = -3 

	# Step 2. Simular data
	x1 = np.random.normal(loc=25, scale=5, size=n) # Trækker x1 fra normalfordeling
	u  = np.random.uniform(low=-50, high=50, size=n) # Trækker u fra uniformfordeling
	x2_star = np.random.uniform(low=10, high=30, size=n) # Trækker x2* fra uniformfordeling
	x2 = rho * x1 + x2_star
	y = beta0 + beta1*x1 + beta2*x2 + u

	## Step 3: Estimer SLR modellen (y ~ x1)
	X = pd.DataFrame({'x1': x1}) 
	X = sm.add_constant(X) 
	SLR_model   = sm.OLS(y,X)
	SLR_results = SLR_model.fit()
	beta1_SLR = SLR_results.params['x1'] # Gemmer beta1 fra SLR-estimatet

	# Step 4: Estimer MLR modellen (y ~ x1 + x2)
	X = pd.DataFrame({'x1': x1, 'x2': x2}) 
	X = sm.add_constant(X) 
	MLR_model   = sm.OLS(y,X)
	MLR_results = MLR_model.fit()
	beta1_MLR = MLR_results.params['x1'] # Gemmer beta1 fra MLR-estimatet

	return beta1_SLR, beta1_MLR

In [18]:
### Denne funktion kører vores simulering 1000 gange og gemmer resultaterne i et DataFrame.
def monte_carlo(reps=1000):
	np.random.seed(67) # Sæt seed så vi får samme tilfældige resultater hver gang
	SLR_results = [] # Liste til at gemme beta1 resultater fra SLR
	MLR_results = [] # Liste til at gemme beta1 resultater fra MLR

	for r in range(reps): # Kør simulationen "reps" antal gange
		beta1_MLR, beta1_SLR = simulate()
		SLR_results.append(beta1_MLR)
		MLR_results.append(beta1_SLR)

	# Saml alle resultaterne i et pandas DataFrame
	results = pd.DataFrame({'beta1_SLR': SLR_results, 'beta1_MLR': MLR_results})

	return results

Kør simulationen og gem resultaterne:

In [19]:
results = monte_carlo()

### Opgave 2 
Sammenlign $\widetilde{\beta}_1$ og $\widehat{\beta}_1$. Stemmer resultaterne overens med det, som I fandt i gruppespørgsmål 1? I kan bruge pandas metoden `.describe()` til at se gennemsnittet og variansen af $\widetilde{\beta}_1$ og $\widehat{\beta}_1$ fra eksperimentet.

**Din kode:**

In [20]:
results.describe()

,beta1_SLR,beta1_MLR
count,1000.000000,1000.000000
mean,0.528726,2.050741
std,0.968835,0.930305
min,-2.675054,-0.821768
25%,-0.139807,1.451671
50%,0.578684,2.014360
75%,1.173017,2.673747
max,3.510390,5.554127


**Dit svar:**

Det er den samme bias da middelværdien er 0.5 som er -1.5 under den sande værdi altså er der pålagt en negativ bias på omkring -1.5

### Opgave 3
Kør et Monte Carlo eksperiment med $n=10$ og $n=100$. Sammenlign $\widetilde{\beta}_1$ og $\widehat{\beta}_1$. Er både $\widetilde{\beta}_1$ og $\widehat{\beta}_1$ konsistente estimatorer for $\beta_1$?

In [27]:
def simulate(x):
	## Step 1. Definer parameterværdier
	n = x
	rho = 0.5
	beta0 = 1
	beta1 = 2
	beta2 = -3 

	# Step 2. Simular data
	x1 = np.random.normal(loc=25, scale=5, size=n) # Trækker x1 fra normalfordeling
	u  = np.random.uniform(low=-50, high=50, size=n) # Trækker u fra uniformfordeling
	x2_star = np.random.uniform(low=10, high=30, size=n) # Trækker x2* fra uniformfordeling
	x2 = rho * x1 + x2_star
	y = beta0 + beta1*x1 + beta2*x2 + u

	## Step 3: Estimer SLR modellen (y ~ x1)
	X = pd.DataFrame({'x1': x1}) 
	X = sm.add_constant(X) 
	SLR_model   = sm.OLS(y,X)
	SLR_results = SLR_model.fit()
	beta1_SLR = SLR_results.params['x1'] # Gemmer beta1 fra SLR-estimatet

	# Step 4: Estimer MLR modellen (y ~ x1 + x2)
	X = pd.DataFrame({'x1': x1, 'x2': x2}) 
	X = sm.add_constant(X) 
	MLR_model   = sm.OLS(y,X)
	MLR_results = MLR_model.fit()
	beta1_MLR = MLR_results.params['x1'] # Gemmer beta1 fra MLR-estimatet

	return beta1_SLR, beta1_MLR


In [30]:
def monte_carlo(reps=1000, n=10):
	np.random.seed(67) # Sæt seed så vi får samme tilfældige resultater hver gang
	SLR_results = [] # Liste til at gemme beta1 resultater fra SLR
	MLR_results = [] # Liste til at gemme beta1 resultater fra MLR

	for r in range(reps): # Kør simulationen "reps" antal gange
		beta1_MLR, beta1_SLR = simulate(n)
		SLR_results.append(beta1_MLR)
		MLR_results.append(beta1_SLR)

	# Saml alle resultaterne i et pandas DataFrame
	results = pd.DataFrame({'beta1_SLR': SLR_results, 'beta1_MLR': MLR_results})

	return results
results1 = monte_carlo()
print ('n = 10')
print(results1.describe())

def monte_carlo(reps=1000, n=100):
	np.random.seed(67) # Sæt seed så vi får samme tilfældige resultater hver gang
	SLR_results = [] # Liste til at gemme beta1 resultater fra SLR
	MLR_results = [] # Liste til at gemme beta1 resultater fra MLR

	for r in range(reps): # Kør simulationen "reps" antal gange
		beta1_MLR, beta1_SLR = simulate(n)
		SLR_results.append(beta1_MLR)
		MLR_results.append(beta1_SLR)

	# Saml alle resultaterne i et pandas DataFrame
	results = pd.DataFrame({'beta1_SLR': SLR_results, 'beta1_MLR': MLR_results})

	return results
results2 = monte_carlo()
print ('n = 100')
print(results2.describe())


n = 10
         beta1_SLR    beta1_MLR
count  1000.000000  1000.000000
mean      0.482234     1.996577
std       2.561175     2.540690
min      -8.478966   -13.465256
25%      -1.217961     0.363537
50%       0.483445     1.991169
75%       2.134397     3.642434
max       9.899103    11.164804
n = 100
         beta1_SLR    beta1_MLR
count  1000.000000  1000.000000
mean      0.498609     2.009082
std       0.655088     0.620320
min      -2.022556     0.099329
25%       0.067699     1.591525
50%       0.508627     1.997662
75%       0.925811     2.404796
max       2.485817     3.992652


**Dit svar:**

De er begge middelret, men standartfejlen er meget større når n = 10 end når n = 100. Men de giver stadig samme estimat og er konsistente. 

### Opgave 4
Kør er Monte Carlo eksperiment med $\rho=1$ og $\rho=0$ (og $n = 50$). Er multikolinearitet et problem, når $\rho = 1$? Bliver den asymptotiske bias større eller mindre? Hvis den gør, hvorfor er det tilfældet? Hvilken forskel er der i variansen af $\widetilde{\beta}_1$ og $\widehat{\beta}_1$, når $\rho = 0$? Hvilken estimator vil I foretrække?

**Dit svar:**

### Opgave 5
Kør et Monte Carlo eksperiment med $\rho=0.5$ og $\beta_2=0$. Sammenlign $\widetilde{\beta}_1$ og $\widehat{\beta}_1$. Hvilken er estimator vil I foretrække i dette tilfælde?

**Dit svar:**

### Opgave 6
Kør det oprindelige Monte Carlo eksperiment med et andet seed-nummer. I kan f.eks. sætte seed-nummeret til det år, I er født. Hvad kan I konkludere om $\widetilde{\beta}_1$ og $\widehat{\beta}_1$ ud fra denne ændring?

**Dit svar:**

### Ekstraopgave: 
Hvis I har mere tid tilbage, kan I lave histogrammer af  $\widetilde{\beta}_1$ og $\widehat{\beta}_1$  for hver ændring i Monte Carlo eksperimentet fra spørgsmål 2 til 6. I kan bruge koden nedenfor. Husk at gemme histogrammerne efter hver simulation, hvis I gerne vil sammenligne med det oprindelige eksperiment.

```py
import seaborn as sns
sns.histplot(results);
```

**Din kode:**